# Model Pipeline: ISYE 6740 Final Project

In [2]:
#==============================================#
# IMPORT MODULES
#==============================================#
import pandas as pd
import numpy as np
import openpyxl
from iosacal import R, combine, iplot
import matplotlib.pyplot as plt


Matplotlib is building the font cache; this may take a moment.


## Data Acquisition
Data Acquisition. The dataset consists of ~475 radiocarbon measurements collected over 50 years of archaeological fieldwork by the Smithsonian Institution. The radiocarbon assays include geographic metadata, sample material, and cultural attribution.

To reduce the risk of exposing sensitive archaeological site locations and preserve the integrity of these irreplaceable cultural resources, all site spatial coordinates will be perturbed or aggregated prior to inclusion in the report and model pipeline. This was accomplished prior to loading the dataset. 



In [3]:
#==============================================#
# IMPORT AND CLEAN DATA
#==============================================#
radiocarbon = pd.read_excel("radiocarbon.xlsx")

# Remove samples from none-cultural sources (e.g., sediment cores).
# Samples with "Geological Sample" or "Pollen Core" in "GenProv" are considered non-cultural.
radiocarbon = radiocarbon[~radiocarbon['GenProv']
                .str.contains("POLLEN CORE|GEOLOGICAL SAMPLE", case=False)]

print("Number of cultural samples available:", radiocarbon.shape[0])

Number of cultural samples available: 370


## Date Calibration. 

Radiocarbon dates will be calibrated to correct for temporal fluctuations in background atmospheric carbon-14 using the IntCal20 calibration curves (Reimer, 2020). Samples derived from terrestrial and marine sources must be calibrated separately to avoid carbon reservoir effects.

In [ ]:
#==============================================#
# CALIBRATE RADIOCARBON DATA
#==============================================#
# Calibrate radiocarbon samples to calendar years based on samples of known age.
# Calibration is conducted using IOSACAL package: https://c14.iosa.it/en/latest/.
# Marine and terrestrial data should be calibrated seperately.
# Assumes that dates not marked as from a marine source are terrestrial.

# Identify marine dates
# keywords: marine, shell, fat, whale, seal
marine_radiocarbon = radiocarbon[radiocarbon['C14SampTyp']
                .str.contains("marine|shell|fat|whale|seal", case=False)]

# Identify terrestrial dates
terrestrial_radiocarbon = radiocarbon[~ radiocarbon['C14SampTyp']
                .str.contains("marine|shell|fat|whale|seal", case=False)]

# Calibrate terrestrial samples according to IntCal20 calibration curve
# Create combined date object for calibration
terrestrial_cals = []
for row in terrestrial_radiocarbon.itertuples(index=True):
    uncal = R(row.C14Date1, row.C14Vari1, row.C14LabNo1) 
    cal = uncal.calibrate("intcal20")
    terrestrial_cals.append(cal)

# Calibrate marine samples according to Marine20 calibration curve
marine_cals = []
for row in marine_radiocarbon.itertuples(index=True):
    uncal = R(row.C14Date1, row.C14Vari1, row.C14LabNo1) 
    cal = uncal.calibrate("marine20")
    marine_cals.append(cal)

# The calibrated dates should probably actually be in a multidimension numpy array for improved plotting

## Oversampling Corrections. 

Dates-as-data can lead to biased estimates of population density due to intensive focus on certain productive archaeological sites. To reduce this bias, samples may be aggregated by site and temporal phase so as not to oversample key locations, following techniques commonly applied in archaeological demography (Crema 2018; Shennan et al., 2013). This methodology is still under discussion in the literature and may lack strong theoretical grounding.

In [ ]:
# Attack this later. Let's look at model estimation now. 
# https://www.cambridge.org/core/journals/cambridge-archaeological-journal/article/archaeological-and-genetic-foundations-of-the-european-population-during-the-late-glacial-implications-for-agricultural-thinking/1838A9994BEF630FE231867676CB3728
# https://www.sciencedirect.com/science/article/pii/S0305440306002330?via%3Dihub#bib12



## Mixture Model Estimation

Gaussian Mixture Model Estimates. Gaussian mixture models will be fit according to the procedure outlined in Price et al. (2018).

In [ ]:
#==============================================#
# BAYESIAN ESTIMATION OF GAUSSIAN MIXTURE MODEL
#==============================================#
# PyMC used for mixture model training 
import pymc as pm
print(f"PyMC version:{pm.__version__}")



1.13.1


# Posterior Predictive Checks.

Posterior predictive checks will be used to assess the agreement between the estimated and observed radiocarbon distribution. Comparisons may include overall shape, variance, and other summary statistics.


## Out-of-Sample Likelihood. 

Leave-one-out cross-validation will be used to estimate the model’s predictive performance. Due to the computational complexity of refitting multiple MCMC models, approximate approaches, such as Pareto-smoothed importance sampling leave-one-out, may be used if appropriate (Vehtari et al., 2018).


## Oversampling Sensitivity Check

See how much estimates change with/without oversampling correction. 

## Comparison Against Archaeological Literature.

Population estimates will be compared against established cultural chronologies from the archaeological literature (e.g., Fitzhugh, 1985). Agreement with known phases and transitions will provide a preliminary check on the validity of the model.
